# RealBasicVSR — Complete Zero → 100% Master

এটি নতুন Colab account/runtime-এ পুরো workflow চালানোর জন্য:
**Drive → environment → model → AI frames → Drive checkpoint → MP4 → original audio → final Drive output**

প্রতিটি frame Google Drive-এ save হবে। তাই মাঝপথে runtime বন্ধ হলেও frame পুনরায় তৈরি করার বদলে পরের run-এ missing frame থেকে কাজ চালানো যাবে.

In [2]:

# STEP 0 — GPU STATUS CHECK
# শুধু GPU-এর বর্তমান অবস্থা দেখবে
# কোনো package install বা পরিবর্তন করবে না

import torch

print("========================================")
print("GPU STATUS CHECK")
print("========================================")

print("PyTorch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("GPU: NOT AVAILABLE")

print("========================================")

GPU STATUS CHECK
PyTorch: 2.11.0+cpu
CUDA version: None
CUDA available: False
GPU: NOT AVAILABLE


In [ ]:
# STEP 1 — Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# STEP 2 — Multi-video project folders

import os

PROJECT = "RealBasicVSR_Project"
DRIVE_ROOT = f"/content/drive/MyDrive/{PROJECT}"

INPUT_DIR = f"{DRIVE_ROOT}/input"
FRAMES_DIR = f"{DRIVE_ROOT}/frames"
RESULTS_DIR = f"{DRIVE_ROOT}/results"

for p in [INPUT_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(p, exist_ok=True)

print("Drive Project:", DRIVE_ROOT)
print("Input folder:", INPUT_DIR)
print("Frames folder:", FRAMES_DIR)
print("Results folder:", RESULTS_DIR)

Drive Project: /content/drive/MyDrive/RealBasicVSR_Project
Input folder: /content/drive/MyDrive/RealBasicVSR_Project/input
Frames folder: /content/drive/MyDrive/RealBasicVSR_Project/frames
Results folder: /content/drive/MyDrive/RealBasicVSR_Project/results


In [ ]:

# MASTER SETUP — STEP 3 FIXED
# Install Python 3.10 venv and create virtual environment

!apt-get update -y
!apt-get install -y python3.10-venv

!rm -rf /content/rbvsr_env
!python3.10 -m venv /content/rbvsr_env

!/content/rbvsr_env/bin/python --version
!/content/rbvsr_env/bin/pip --version

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [114 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [3,045 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntug

In [ ]:
# STEP 4 — Install pip 22.0.2

!/content/rbvsr_env/bin/python -m pip install --upgrade "pip==22.0.2"
!/content/rbvsr_env/bin/pip --version

pip 22.0.2 from /content/rbvsr_env/lib/python3.10/site-packages/pip (python 3.10)


In [ ]:
# STEP 5 — Install NumPy 1.26.4

!/content/rbvsr_env/bin/python -m pip install "numpy==1.26.4"
!/content/rbvsr_env/bin/python -c "import numpy; print('NumPy:', numpy.__version__)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 65.2 MB/s eta 0:00:00
NumPy: 1.26.4


In [ ]:
# STEP 6 — PyTorch 1.13.1 + CUDA 11.7

!/content/rbvsr_env/bin/python -m pip install "torch==1.13.1+cu117" "torchvision==0.14.1+cu117" --extra-index-url https://download.pytorch.org/whl/cu117

!/content/rbvsr_env/bin/python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available())"

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu117
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 973.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 54.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 KB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 KB 17.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 KB 15.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.1/271.1 KB 45.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 KB 32.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 KB 31.4 MB/s eta 0:00:00
PyTorch: 1.13.1+cu117
CUDA: 11.7
CUDA available: True


In [ ]:
# STEP 7 — Torchvision verification

!/content/rbvsr_env/bin/python -c "import torchvision; print('Torchvision:', torchvision.__version__)"

Torchvision: 0.14.1+cu117


In [ ]:
# STEP 8 — Install OpenMIM

!/content/rbvsr_env/bin/python -m pip install "openmim"

!/content/rbvsr_env/bin/python -c "import mim; print('OpenMIM: OK')"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 KB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 KB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 KB 27.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 KB 69.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 KB 25.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 KB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 115.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 KB 48.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.5/347.5 KB 55.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 KB 42.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 KB 62.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:

# MASTER SETUP — STEP 9
# Install pre-built MMCV-full 1.7.2 wheel
# Keep dependency versions controlled

!/content/rbvsr_env/bin/python -m pip install \
    "addict==2.4.0" \
    "yapf==0.43.0" \
    "tomli==2.4.1" \
    "platformdirs==4.12.2" \
    --no-deps

!/content/rbvsr_env/bin/python -m pip install \
    "mmcv-full==1.7.2" \
    --no-deps \
    https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/mmcv_full-1.7.2-cp310-cp310-manylinux1_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 MB 13.1 MB/s eta 0:00:00
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/rbvsr_env/lib/python3.10/site-packages/mmcv/__init__.py", line 6, in <module>
    from .fileio import *
  File "/content/rbvsr_env/lib/python3.10/site-packages/mmcv/fileio/__init__.py", line 2, in <module>
    from .file_client import BaseStorageBackend, FileClient
  File "/content/rbvsr_env/lib/python3.10/site-packages/mmcv/fileio/file_client.py", line 15, in <module>
    from mmcv.utils.misc import has_method
  File "/content/rbvsr_env/lib/python3.10/site-packages/mmcv/utils/__init__.py", line 3, in <module>
    from .config import Config, ConfigDict, DictAction
  File "/content/rbvsr_env/lib/python3.10/site-packages/mmcv/utils/config.py", line 18, in <module>
    import yapf
  File "/content/rbvsr_env/lib/python3.10/site-packages/yapf/__init__.py", line 38, in <module>
    from yapf.yapflib import file_resources
 

In [ ]:

# STEP 10 — Install OpenCV 4.8.1.78

!/content/rbvsr_env/bin/python -m pip install --force-reinstall "opencv-python==4.8.1.78" --no-deps

!/content/rbvsr_env/bin/python -c "import cv2, numpy; print('OpenCV:', cv2.__version__); print('NumPy:', numpy.__version__)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 MB 21.6 MB/s eta 0:00:00
OpenCV: 4.8.1
NumPy: 1.26.4


In [ ]:

# STEP 11 — Install MMEditing 0.16.1

!/content/rbvsr_env/bin/python -m pip install "mmedit==0.16.1" --no-deps

!/content/rbvsr_env/bin/python -c "import mmedit; print('MMEditing:', mmedit.__version__)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.8/606.8 KB 10.2 MB/s eta 0:00:00
/content/rbvsr_env/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
MMEditing: 0.16.1


In [ ]:

# STEP 12A — Install SciPy for RealBasicVSR

!/content/rbvsr_env/bin/python -m pip install "scipy==1.10.1" --no-deps

!/content/rbvsr_env/bin/python -c "import scipy, numpy; print('SciPy:', scipy.__version__); print('NumPy:', numpy.__version__)"

SciPy: 1.10.1
NumPy: 1.26.4


In [ ]:

# STEP 12B — Install tqdm

!/content/rbvsr_env/bin/python -m pip install "tqdm==4.66.5" --no-deps

!/content/rbvsr_env/bin/python -c "import tqdm; print('tqdm:', tqdm.__version__)"

tqdm: 4.66.5


In [ ]:

# STEP 13 — RealBasicVSR source

!rm -rf /content/RealBasicVSR
!git clone https://github.com/ckkelvinchan/RealBasicVSR.git /content/RealBasicVSR

!test -f /content/RealBasicVSR/configs/realbasicvsr_c64b20_1x30x8_lr5e-5_150k_reds.py && echo "RealBasicVSR source: OK"

Cloning into '/content/RealBasicVSR'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 164 (delta 34), reused 23 (delta 23), pack-reused 113 (from 1)
Receiving objects: 100% (164/164), 25.97 MiB | 45.84 MiB/s, done.
Resolving deltas: 100% (50/50), done.
RealBasicVSR source: OK


In [ ]:

# STEP 14 — Download RealBasicVSR checkpoint

!mkdir -p /content/RealBasicVSR/checkpoints

!wget -O /content/RealBasicVSR/checkpoints/RealBasicVSR.pth \
"https://www.dropbox.com/s/eufigxmmkv5woop/RealBasicVSR.pth?dl=1"

!ls -lh /content/RealBasicVSR/checkpoints/RealBasicVSR.pth

--2026-10-02 06:03:21--  https://www.dropbox.com/s/eufigxmmkv5woop/RealBasicVSR.pth?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/gt6l0t9ojxf0uswhjbc6q/RealBasicVSR.pth?rlkey=75xiwo2ohsmk0pxp8ozp4wm3v&dl=1 [following]
--2026-10-02 06:03:21--  https://www.dropbox.com/scl/fi/gt6l0t9ojxf0uswhjbc6q/RealBasicVSR.pth?rlkey=75xiwo2ohsmk0pxp8ozp4wm3v&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc799e266f0e04ae39b72b1f5d02.dl.dropboxusercontent.com/cd/0/inline/DJNwTB8cposOMVK2MjdiXIgoKmEiTJy-PJcFi5U6gB6gF0PSKO70cR8Zu_bjMnYFfjquzVj6gx05jsl4DsK6o37rD0sOkM3lgfq44UCWIz1OVwwfd1fESTpO5NXIyazdS2c/file?dl=1# [following]
--2026-10-02 06:03:21--  https://uc799e266f0e04ae39b72b1f5d02.dl.dropboxusercontent.com/cd

In [ ]:

# STEP 15 — Download VGG19 pretrained weights

!/content/rbvsr_env/bin/python -c "import torch; from torchvision.models import vgg19, VGG19_Weights; vgg19(weights=VGG19_Weights.IMAGENET1K_V1); print('VGG19: OK')"

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100% 548M/548M [00:02<00:00, 246MB/s]
VGG19: OK


In [ ]:

# STEP 16 — # FINAL MASTER SETUP CHECK — CORRECT ENVIRONMENT

!/content/rbvsr_env/bin/python -c "import sys,numpy,torch,torchvision,cv2,mmcv,mmedit,tqdm,scipy; print('Python:',sys.version.split()[0],'environment OK'); print('NumPy:',numpy.__version__); print('PyTorch:',torch.__version__); print('CUDA:',torch.version.cuda); print('CUDA available:',torch.cuda.is_available()); print('Torchvision:',torchvision.__version__); print('OpenCV:',cv2.__version__); print('MMCV:',mmcv.__version__); print('MMEditing:',mmedit.__version__); print('SciPy:',scipy.__version__); print('tqdm:',tqdm.__version__); print('MASTER SETUP: OK')"

/content/rbvsr_env/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
Python: 3.10.12 environment OK
NumPy: 1.26.4
PyTorch: 1.13.1+cu117
CUDA: 11.7
CUDA available: True
Torchvision: 0.14.1+cu117
OpenCV: 4.8.1
MMCV: 1.7.2
MMEditing: 0.16.1
SciPy: 1.10.1
tqdm: 4.66.5
MASTER SETUP: OK


In [ ]:

# VIDEO WORKFLOW — STEP 1
# Check input videos

import os

INPUT_DIR = "/content/drive/MyDrive/RealBasicVSR_Project/input"

videos = [
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith((".mp4", ".mkv", ".avi", ".mov", ".webm"))
]

print("Input videos:", len(videos))

for i, video in enumerate(videos, 1):
    size_mb = os.path.getsize(os.path.join(INPUT_DIR, video)) / (1024 * 1024)
    print(f"{i}. {video} — {size_mb:.2f} MB")

Input videos: 2
1. 144p_40s_২৭_কোটি.mp4 — 0.88 MB
2. 0Boruto_272cuts_144p.mp4 — 0.76 MB


In [ ]:

# VIDEO WORKFLOW — STEP 1–3 COMBINED
# Video Select + Information + Project Folders

import os
import subprocess
import json

INPUT_DIR = "/content/drive/MyDrive/RealBasicVSR_Project/input"
FRAMES_ROOT = "/content/drive/MyDrive/RealBasicVSR_Project/frames"
RESULTS_ROOT = "/content/drive/MyDrive/RealBasicVSR_Project/results"

# Find videos
videos = sorted([
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith((".mp4", ".mkv", ".avi", ".mov", ".webm"))
])

print("Available videos:")
for i, video in enumerate(videos, 1):
    size_mb = os.path.getsize(os.path.join(INPUT_DIR, video)) / (1024 * 1024)
    print(f"{i}. {video} — {size_mb:.2f} MB")

print()
print("Select video number:")
VIDEO_NUMBER = int(input())

if VIDEO_NUMBER < 1 or VIDEO_NUMBER > len(videos):
    raise ValueError("Invalid video number.")

# Selected video
selected_video = videos[VIDEO_NUMBER - 1]
INPUT_VIDEO = os.path.join(INPUT_DIR, selected_video)

# Video-specific folders
video_name = os.path.splitext(selected_video)[0]

VIDEO_FRAMES_DIR = os.path.join(FRAMES_ROOT, video_name)
VIDEO_RESULTS_DIR = os.path.join(RESULTS_ROOT, video_name)

os.makedirs(VIDEO_FRAMES_DIR, exist_ok=True)
os.makedirs(VIDEO_RESULTS_DIR, exist_ok=True)

# Read video information with ffprobe
probe_cmd = [
    "ffprobe", "-v", "error",
    "-show_streams",
    "-show_format",
    "-of", "json",
    INPUT_VIDEO
]

probe = subprocess.run(
    probe_cmd,
    capture_output=True,
    text=True,
    check=True
)

info = json.loads(probe.stdout)

video_stream = next(
    s for s in info["streams"]
    if s["codec_type"] == "video"
)

# Resolution
width = video_stream.get("width")
height = video_stream.get("height")

# FPS
fps_text = video_stream.get("r_frame_rate", "0/1")
fps_num, fps_den = map(int, fps_text.split("/"))
fps = fps_num / fps_den if fps_den else 0

# Duration
duration = float(
    video_stream.get(
        "duration",
        info.get("format", {}).get("duration", 0)
    )
)

# Frame count
frame_count = video_stream.get("nb_frames")

if frame_count is None:
    frame_probe_cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-count_frames",
        "-show_entries", "stream=nb_read_frames",
        "-of", "default=noprint_wrappers=1:nokey=1",
        INPUT_VIDEO
    ]

    frame_result = subprocess.run(
        frame_probe_cmd,
        capture_output=True,
        text=True,
        check=True
    )

    frame_count = frame_result.stdout.strip()

# Audio check
audio_streams = [
    s for s in info["streams"]
    if s["codec_type"] == "audio"
]

audio_status = "Yes" if audio_streams else "No"

# Final information
print()
print("========== SELECTED VIDEO ==========")
print("Selected video:", selected_video)
print("Input path:", INPUT_VIDEO)
print()
print("Resolution:", f"{width}x{height}")
print("FPS:", f"{fps:.3f}")
print("Duration:", f"{duration:.2f} seconds")
print("Frames:", frame_count)
print("Audio:", audio_status)
print()
print("Frames folder:", VIDEO_FRAMES_DIR)
print("Results folder:", VIDEO_RESULTS_DIR)
print()
print("Selection applied: OK")
print("Project folders: OK")
print("Video information: OK")

Available videos:
1. 0Boruto_272cuts_144p.mp4 — 0.76 MB
2. 144p_40s_২৭_কোটি.mp4 — 0.88 MB

Select video number:
1

========== SELECTED VIDEO ==========
Selected video: 0Boruto_272cuts_144p.mp4
Input path: /content/drive/MyDrive/RealBasicVSR_Project/input/0Boruto_272cuts_144p.mp4

Resolution: 254x144
FPS: 25.000
Duration: 30.00 seconds
Frames: 750
Audio: Yes

Frames folder: /content/drive/MyDrive/RealBasicVSR_Project/frames/0Boruto_272cuts_144p
Results folder: /content/drive/MyDrive/RealBasicVSR_Project/results/0Boruto_272cuts_144p

Selection applied: OK
Project folders: OK
Video information: OK


In [ ]:

# VIDEO WORKFLOW — STEP 4 UPDATED
# Extract video frames with live progress

import os
import subprocess
import time

os.makedirs(VIDEO_FRAMES_DIR, exist_ok=True)

def count_frames():
    return len([
        f for f in os.listdir(VIDEO_FRAMES_DIR)
        if f.lower().endswith(".png")
    ])

existing_frames = count_frames()

print("========================================")
print("FRAME EXTRACTION")
print("========================================")
print("Video:", selected_video)
print("Frame folder:", VIDEO_FRAMES_DIR)
print("Existing frames:", existing_frames)
print()

if existing_frames > 0:
    print("Existing frames found.")
    print("Frame extraction skipped.")
    print("Existing frame count:", existing_frames)

else:
    print("Starting frame extraction...")
    print("Please wait...")
    print()

    process = subprocess.Popen([
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        "-i", INPUT_VIDEO,
        os.path.join(VIDEO_FRAMES_DIR, "%06d.png")
    ])

    last_count = -1

    while process.poll() is None:
        current_count = count_frames()

        if current_count != last_count:
            print(
                f"\rFrames extracted: {current_count}",
                end="",
                flush=True
            )
            last_count = current_count

        time.sleep(0.5)

    # Final count
    final_count = count_frames()

    print()
    print()
    print("Frame extraction completed.")
    print("Frames extracted:", final_count)

print()
print("========================================")
print("FINAL CHECK")
print("========================================")

final_count = count_frames()

print("Total PNG frames:", final_count)
print("Frame folder:", VIDEO_FRAMES_DIR)

if final_count > 0:
    print("Frame extraction: OK")
else:
    print("Frame extraction: FAILED")

FRAME EXTRACTION
Video: 0Boruto_272cuts_144p.mp4
Frame folder: /content/drive/MyDrive/RealBasicVSR_Project/frames/0Boruto_272cuts_144p
Existing frames: 750

Existing frames found.
Frame extraction skipped.
Existing frame count: 750

FINAL CHECK
Total PNG frames: 750
Frame folder: /content/drive/MyDrive/RealBasicVSR_Project/frames/0Boruto_272cuts_144p
Frame extraction: OK


In [ ]:

# VIDEO WORKFLOW — STEP 4 CHECK
# Verify all extracted frames are complete and valid

import os
from PIL import Image

print("========================================")
print("FRAME INTEGRITY CHECK")
print("========================================")

expected_frames = 750
missing_frames = []
zero_size_frames = []
invalid_frames = []

for i in range(1, expected_frames + 1):

    filename = f"{i:06d}.png"
    filepath = os.path.join(VIDEO_FRAMES_DIR, filename)

    # Missing file
    if not os.path.exists(filepath):
        missing_frames.append(filename)
        continue

    # Zero-byte file
    if os.path.getsize(filepath) == 0:
        zero_size_frames.append(filename)
        continue

    # Validate PNG
    try:
        with Image.open(filepath) as img:
            img.verify()
    except Exception:
        invalid_frames.append(filename)

# Final result
actual_frames = len([
    f for f in os.listdir(VIDEO_FRAMES_DIR)
    if f.lower().endswith(".png")
])

print("Expected frames:", expected_frames)
print("Actual PNG frames:", actual_frames)
print("Missing frames:", len(missing_frames))
print("Zero-size frames:", len(zero_size_frames))
print("Invalid frames:", len(invalid_frames))

print()

if (
    actual_frames == expected_frames
    and not missing_frames
    and not zero_size_frames
    and not invalid_frames
):
    print("========================================")
    print("FRAME CHECK: PASSED")
    print("All 750 frames are present and valid.")
    print("Ready for AI processing.")
    print("========================================")
else:
    print("========================================")
    print("FRAME CHECK: FAILED")
    print("Do NOT start AI processing yet.")
    print("========================================")

    if missing_frames:
        print("Missing:", missing_frames[:20])

    if zero_size_frames:
        print("Zero-size:", zero_size_frames[:20])

    if invalid_frames:
        print("Invalid:", invalid_frames[:20])

FRAME INTEGRITY CHECK
Expected frames: 750
Actual PNG frames: 750
Missing frames: 0
Zero-size frames: 0
Invalid frames: 0

FRAME CHECK: PASSED
All 750 frames are present and valid.
Ready for AI processing.


In [ ]:

# 🔴 FINAL — RealBasicVSR AI Processing
# 750 frames with max_seq_len=10

import os
import subprocess

AI_OUTPUT_DIR = os.path.join(
    VIDEO_RESULTS_DIR,
    "upscaled_frames"
)

os.makedirs(AI_OUTPUT_DIR, exist_ok=True)

input_frames = sorted([
    f for f in os.listdir(VIDEO_FRAMES_DIR)
    if f.lower().endswith(".png")
])

print("========================================")
print("REALBASICVSR FINAL PROCESSING")
print("========================================")
print("Input frames:", len(input_frames))
print("Input folder:", VIDEO_FRAMES_DIR)
print("Output folder:", AI_OUTPUT_DIR)
print("Max sequence length: 10")
print()

if len(input_frames) != 750:
    raise RuntimeError(
        f"Expected 750 input frames, but found {len(input_frames)}."
    )

print("750 input frames confirmed.")
print("Starting RealBasicVSR...")
print("Please keep Colab connected.")
print()

cmd = [
    "/content/rbvsr_env/bin/python",
    "/content/RealBasicVSR/inference_realbasicvsr.py",
    "--max_seq_len", "10",
    "/content/RealBasicVSR/configs/realbasicvsr_c64b20_1x30x8_lr5e-5_150k_reds.py",
    "/content/RealBasicVSR/checkpoints/RealBasicVSR.pth",
    VIDEO_FRAMES_DIR,
    AI_OUTPUT_DIR
]

result = subprocess.run(cmd)

print()
print("========================================")

if result.returncode == 0:

    output_frames = len([
        f for f in os.listdir(AI_OUTPUT_DIR)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

    print("REALBASICVSR PROCESSING: COMPLETED")
    print("Output frames:", output_frames)
    print("========================================")

else:
    print("REALBASICVSR PROCESSING: STOPPED")
    print("Return code:", result.returncode)
    print("========================================")

NameError: name 'VIDEO_RESULTS_DIR' is not defined

## STEP 3 — Put the input MP4 in Drive

প্রথমবার শুধু আপনার MP4 রাখুন:

`My Drive / RealBasicVSR_Project / input`

তারপর নিচের cell চালান।

In [ ]:
# STEP 3 — Select the first MP4 in the input folder
import glob, os
videos = sorted(glob.glob(INPUT_DIR + "/*.mp4") + glob.glob(INPUT_DIR + "/*.MP4"))
if not videos:
    raise FileNotFoundError("Drive input folder-এ MP4 নেই। ভিডিওটি সেখানে upload করুন।")
INPUT_VIDEO = videos[0]
print("Input:", INPUT_VIDEO)
!ffprobe -v error -select_streams v:0 -show_entries stream=codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 "$INPUT_VIDEO"

Input: /content/drive/MyDrive/RealBasicVSR_Project/input/0Boruto_272cuts_144p.mp4
codec_name=h264
width=254
height=144
r_frame_rate=25/1
duration=30.000000


In [ ]:
# STEP 4 — Exact proven Python 3.10 environment
!sudo apt-get update -qq
!sudo apt-get install -y python3.10 python3.10-venv
!rm -rf /content/rbvsr_legacy
!python3.10 -m venv /content/rbvsr_legacy
!/content/rbvsr_legacy/bin/python -m pip install --upgrade pip==22.0.2
!/content/rbvsr_legacy/bin/python -m pip install numpy==1.26.4
!/content/rbvsr_legacy/bin/python -m pip install torch==1.13.1+cu117 torchvision==0.14.1 --extra-index-url https://download.pytorch.org/whl/cu117
!/content/rbvsr_legacy/bin/python -m pip install openmim==0.3.9
!/content/rbvsr_legacy/bin/python -m pip install mmcv-full==1.7.2
!/content/rbvsr_legacy/bin/python -m pip install numpy==1.26.4 opencv-python==4.8.1.78
!/content/rbvsr_legacy/bin/python -m pip install mmedit==0.16.1 tqdm==4.65.2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.10 is already the newest version (3.10.12-1~22.04.18).
python3.10 set to manually installed.
python3.10-venv is already the newest version (3.10.12-1~22.04.18).
0 upgraded, 0 newly installed, 0 to remove and 231 not upgraded.
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu117
  Using cached https://download-r2.pytorch.org/whl/cu117/torch-1.13.1%2Bcu117-cp310-cp310-linux_x86_64.whl (1801.8 MB)
  Using cached https://download-r2.pytorch.org/whl/cu117/torchvision-0.14.1%2Bcu117-cp310-cp310-linux_x86_64.whl (24.3 MB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl 

In [ ]:

!sudo apt-get update -qq
!sudo apt-get install -y python3.10 python3.10-venv
!rm -rf /content/rbvsr_legacy
!python3.10 -m venv /content/rbvsr_legacy

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3-pip-whl python3-setuptools-whl python3.10-minimal
Suggested packages:
  python3.10-doc binfmt-support
The following NEW packages will be installed:
  python3-pip-whl python3-setuptools-whl python3.10-venv
The following packages will be upgraded:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3.10 python3.10-minimal
6 upgraded, 3 newly installed, 0 to remove and 231 not upgraded.
Need to get 14.6 MB of archives.
After this operation, 2,776 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-update

In [ ]:
!/content/rbvsr_legacy/bin/python -m pip install --upgrade pip==22.0.2

In [ ]:
!/content/rbvsr_legacy/bin/python -m pip install numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 62.1 MB/s eta 0:00:00


In [ ]:
!/content/rbvsr_legacy/bin/python -m pip install torch==1.13.1+cu117 torchvision==0.14.1 --extra-index-url https://download.pytorch.org/whl/cu117

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu117
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 962.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 73.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 KB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 KB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.6/261.6 KB 46.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 KB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 KB 27.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 KB 23.9 MB/s eta 0:00:00


In [ ]:
!/content/rbvsr_legacy/bin/python -m pip install openmim==0.3.9

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 KB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 KB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 37.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 KB 51.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 KB 15.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 KB 60.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 51.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 KB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 KB 34.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.5/347.5 KB 25.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 KB 25.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 KB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!/content/rbvsr_legacy/bin/python -m pip install --force-reinstall numpy==1.26.4 opencv-python==4.8.1.78

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 MB 12.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [ ]:
!/content/rbvsr_legacy/bin/python -m pip install mmcv-full==1.7.2 -f https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/index.html

Looking in links: https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/index.html
DEPRECATION: The HTML index page being used (https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/index.html) is not a proper HTML 5 document. This is in violation of PEP 503 which requires these pages to be well-formed HTML 5 documents. Please reach out to the owners of this index page, and ask them to update this index page to a valid HTML 5 document. pip 22.2 will enforce this behaviour change. Discussion can be found at https://github.com/pypa/pip/issues/10825
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 MB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 KB 2.9 MB/s eta 0:00:00


In [ ]:
!/content/rbvsr_legacy/bin/python -c "import torch, mmcv, numpy, cv2; print('Torch:', torch.__version__); print('MMCV:', mmcv.__version__); print('NumPy:', numpy.__version__); print('OpenCV:', cv2.__version__); print('CUDA:', torch.cuda.is_available())"

/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
Torch: 1.13.1+cu117
MMCV: 1.7.2
NumPy: 1.26.4
OpenCV: 4.8.1
CUDA: True


In [ ]:
!/content/rbvsr_legacy/bin/python -m pip install mmedit==0.16.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.8/606.8 KB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.9/333.9 KB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.6/36.6 MB 23.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 125.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 KB 14.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 28.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 102.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 KB 33.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 KB 27.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 111.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 KB 55.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.7/228.7 KB 45.3 MB/s et

In [ ]:
!/content/rbvsr_legacy/bin/python -c "import mmedit; print('MMEditing:', mmedit.__version__)"

/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
MMEditing: 0.16.1


In [ ]:
!/content/rbvsr_legacy/bin/python -m pip install tqdm==4.65.2

In [ ]:
!rm -rf /content/RealBasicVSR
!git clone -q https://github.com/ckkelvinchan/RealBasicVSR.git /content/RealBasicVSR

In [ ]:
!ls -la /content/RealBasicVSR

total 112
drwxr-xr-x 7 root root  4096 Sep 30 00:07 .
drwxr-xr-x 1 root root  4096 Sep 30 00:02 ..
drwxr-xr-x 2 root root  4096 Sep 30 00:03 checkpoints
drwxr-xr-x 2 root root  4096 Sep 30 00:02 configs
-rw-r--r-- 1 root root  6526 Sep 30 00:02 crop_sub_images.py
drwxr-xr-x 3 root root  4096 Sep 30 00:02 data
-rw-r--r-- 1 root root  4599 Sep 30 00:02 generate_video_demo.py
drwxr-xr-x 8 root root  4096 Sep 30 00:02 .git
-rw-r--r-- 1 root root  1411 Sep 30 00:02 .gitignore
-rw-r--r-- 1 root root  4840 Sep 30 00:07 inference_realbasicvsr_backup.py
-rw-r--r-- 1 root root  4840 Sep 30 00:02 inference_realbasicvsr.py
-rw-r--r-- 1 root root 11368 Sep 30 00:02 LICENSE
-rw-r--r-- 1 root root  1393 Sep 30 00:02 .pre-commit-config.yaml
-rw-r--r-- 1 root root 18931 Sep 30 00:02 .pylintrc
-rw-r--r-- 1 root root  6176 Sep 30 00:02 README.md
drwxr-xr-x 3 root root  4096 Sep 30 00:02 realbasicvsr
-rw-r--r-- 1 root root   518 Sep 30 00:02 setup.cfg


In [ ]:
!ls -l /content/RealBasicVSR/configs/

total 28
-rw-r--r-- 1 root root 10426 Sep 30 00:02 realbasicvsr_c64b20_1x30x8_lr5e-5_150k_reds.py
-rw-r--r-- 1 root root  9589 Sep 30 00:02 realbasicvsr_wogan_c64b20_2x30x8_lr1e-4_300k_reds.py
-rw-r--r-- 1 root root  1089 Sep 30 00:02 realbasicvsr_x4.py


In [ ]:
!ls -lh /content/drive/MyDrive/RealBasicVSR_Project/

total 12K
drwx------ 2 root root 4.0K Sep 30 00:17 frames
drwx------ 2 root root 4.0K Sep 29 14:55 input
drwx------ 2 root root 4.0K Sep 29 14:55 results


In [ ]:
!mkdir -p /content/RealBasicVSR/checkpoints
!gdown 1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID -O /content/RealBasicVSR/checkpoints/RealBasicVSR.pth

Downloading...
From (original): https://drive.google.com/uc?id=1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID
From (redirected): https://drive.google.com/uc?id=1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID&confirm=t&uuid=f65974bf-fee1-478d-8d87-c7efd9705ba2
To: /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
100% 210M/210M [00:02<00:00, 71.0MB/s]


In [ ]:
!ls -lh /content/RealBasicVSR/checkpoints/RealBasicVSR.pth

-rw-r--r-- 1 root root 201M Nov 30  2021 /content/RealBasicVSR/checkpoints/RealBasicVSR.pth


In [ ]:
!/content/rbvsr_legacy/bin/python -c "from mmedit.apis import init_model, restoration_inference; print('MMEditing API: OK')"

/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
MMEditing API: OK


In [ ]:
!/content/rbvsr_legacy/bin/python -c "import torchvision; from torchvision.models import vgg19; print('Torchvision:', torchvision.__version__); print('VGG19: OK')"

Torchvision: 0.14.1+cu117
VGG19: OK


In [ ]:
!/content/rbvsr_legacy/bin/python -c "from mmedit.apis import init_model; model=init_model('/content/RealBasicVSR/configs/realbasicvsr_x4.py','/content/RealBasicVSR/checkpoints/RealBasicVSR.pth',device='cuda'); print('RealBasicVSR model: OK')"

/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
2026-09-30 00:20:15,944 - mmedit - INFO - load checkpoint from torchvision path: torchvision://vgg19
load checkpoint from local path: /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
RealBasicVSR model: OK


In [ ]:
# Test Frames
!rm -rf /content/test_frames
!mkdir -p /content/test_frames
!ffmpeg -y -i "$INPUT_VIDEO" -vsync 0 /content/test_frames/%05d.png
!find /content/test_frames -type f -name '*.png' | wc -l

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
# Test Upscaled
!rm -rf /content/test_upscaled
!mkdir -p /content/test_upscaled
!/content/rbvsr_legacy/bin/python -c "import cv2, torch; from mmedit.apis import init_model, restoration_inference; model=init_model('/content/RealBasicVSR/configs/realbasicvsr_x4.py','/content/RealBasicVSR/checkpoints/RealBasicVSR.pth',device='cuda'); img=cv2.imread('/content/test_frames/00001.png'); result=restoration_inference(model,img); out=result['output']; out=out.squeeze(0).permute(1,2,0).clamp(0,1).cpu().numpy(); cv2.imwrite('/content/test_upscaled/00001.png', cv2.cvtColor((out*255).astype('uint8'),cv2.COLOR_RGB2BGR)); print('AI upscale test: OK'); print('Output shape:', out.shape)"

/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
2026-09-29 23:15:10,044 - mmedit - INFO - load checkpoint from torchvision path: torchvision://vgg19
load checkpoint from local path: /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/rbvsr_legacy/lib/python3.10/site-packages/mmedit/apis/restoration_inference.py", line 24, in restoration_inference
    for pipeline in list(cfg.test_pipeline):
  File "/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/utils/config.py", line 524, in __getattr__
    return getattr(self

In [ ]:
!cp /content/RealBasicVSR/inference_realbasicvsr.py /content/RealBasicVSR/inference_realbasicvsr_backup.py

In [ ]:
%%writefile /content/RealBasicVSR/inference_frames_safe.py
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from mmcv import Config
from mmedit.apis import init_model, restoration_inference

config_file = "/content/RealBasicVSR/configs/realbasicvsr_x4.py"
checkpoint_file = "/content/RealBasicVSR/checkpoints/RealBasicVSR.pth"

input_dir = "/content/test_frames"
output_dir = "/content/test_upscaled"

os.makedirs(output_dir, exist_ok=True)

model = init_model(config_file, checkpoint_file, device="cuda")
model.eval()

frames = sorted(
    f for f in os.listdir(input_dir)
    if f.lower().endswith(".png")
)

print("Total frames:", len(frames))

for name in tqdm(frames, desc="Upscaling"):
    input_path = os.path.join(input_dir, name)
    output_path = os.path.join(output_dir, name)

    if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
        continue

    img = cv2.imread(input_path)

    with torch.no_grad():
        result_dict = restoration_inference(model, img)
        output = result_dict["output"]

    output = output.squeeze(0).permute(1, 2, 0)
    output = output.clamp(0, 1).cpu().numpy()
    output = (output * 255).astype(np.uint8)
    output = cv2.cvtColor(output, cv2.COLOR_RGB2BGR)

    cv2.imwrite(output_path, output)

print("DONE: frames saved to", output_dir)

Writing /content/RealBasicVSR/inference_frames_safe.py


In [ ]:
!/content/rbvsr_legacy/bin/python /content/RealBasicVSR/inference_frames_safe.py

/content/rbvsr_legacy/bin/python: can't open file '/content/RealBasicVSR/inference_frames_safe.py': [Errno 2] No such file or directory


In [ ]:
%%writefile /content/RealBasicVSR/inference_frames_safe.py
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from mmedit.apis import init_model

config_file = "/content/RealBasicVSR/configs/realbasicvsr_x4.py"
checkpoint_file = "/content/RealBasicVSR/checkpoints/RealBasicVSR.pth"

input_dir = "/content/test_frames"
output_dir = "/content/test_upscaled"

os.makedirs(output_dir, exist_ok=True)

model = init_model(config_file, checkpoint_file, device="cuda")
model.eval()

frames = sorted(
    f for f in os.listdir(input_dir)
    if f.lower().endswith(".png")
)

print("Total frames:", len(frames))

for name in tqdm(frames, desc="Upscaling"):
    input_path = os.path.join(input_dir, name)
    output_path = os.path.join(output_dir, name)

    if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
        continue

    img = cv2.imread(input_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = torch.from_numpy(img / 255.0).permute(2, 0, 1).float()
    img = img.unsqueeze(0).unsqueeze(1).cuda()

    with torch.no_grad():
        result = model(img, test_mode=True)
        output = result["output"].cpu()

    output = output.squeeze(0).squeeze(0).permute(1, 2, 0)
    output = output.clamp(0, 1).numpy()
    output = (output * 255).astype(np.uint8)
    output = cv2.cvtColor(output, cv2.COLOR_RGB2BGR)

    cv2.imwrite(output_path, output)

print("DONE: frames saved to", output_dir)

Writing /content/RealBasicVSR/inference_frames_safe.py


In [ ]:
!/content/rbvsr_legacy/bin/python /content/RealBasicVSR/inference_frames_safe.py

/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
2026-09-30 00:22:31,999 - mmedit - INFO - load checkpoint from torchvision path: torchvision://vgg19
load checkpoint from local path: /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
Traceback (most recent call last):
  File "/content/RealBasicVSR/inference_frames_safe.py", line 20, in <module>
    f for f in os.listdir(input_dir)
FileNotFoundError: [Errno 2] No such file or directory: '/content/test_frames'


In [ ]:
!python3 - <<'PY'
from PIL import Image
img = Image.open('/content/test_upscaled/00001.png')
print("Output size:", img.size)
PY

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')


FileNotFoundError: [Errno 2] No such file or directory: '/content/test_upscaled/00001.png'

In [ ]:
import os

os.makedirs(FRAMES_DIR, exist_ok=True)

print("Drive Frames folder:")
print(FRAMES_DIR)

Drive Frames folder:
/content/drive/MyDrive/RealBasicVSR_Project/frames


In [ ]:
%%writefile /content/RealBasicVSR/inference_drive_resume.py
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from mmedit.apis import init_model

config_file = "/content/RealBasicVSR/configs/realbasicvsr_x4.py"
checkpoint_file = "/content/RealBasicVSR/checkpoints/RealBasicVSR.pth"

input_dir = "/content/test_frames"
output_dir = "/content/drive/MyDrive/RealBasicVSR_Project/frames"

os.makedirs(output_dir, exist_ok=True)

model = init_model(config_file, checkpoint_file, device="cuda")
model.eval()

frames = sorted(
    f for f in os.listdir(input_dir)
    if f.lower().endswith(".png")
)

print("Total frames:", len(frames))

for name in tqdm(frames, desc="AI Upscaling"):
    input_path = os.path.join(input_dir, name)
    output_path = os.path.join(output_dir, name)

    # Already completed frame → skip
    if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
        continue

    img = cv2.imread(input_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = torch.from_numpy(img / 255.0).permute(2, 0, 1).float()
    img = img.unsqueeze(0).unsqueeze(1).cuda()

    with torch.no_grad():
        result = model(img, test_mode=True)
        output = result["output"].cpu()

    output = output.squeeze(0).squeeze(0).permute(1, 2, 0)
    output = output.clamp(0, 1).numpy()
    output = (output * 255).astype(np.uint8)
    output = cv2.cvtColor(output, cv2.COLOR_RGB2BGR)

    cv2.imwrite(output_path, output)

print("DONE: AI frames saved to Drive")

Writing /content/RealBasicVSR/inference_drive_resume.py


In [ ]:
import os

test_file = f"{FRAMES_DIR}/test_write.txt"

with open(test_file, "w") as f:
    f.write("Drive write test OK")

print("Created:", test_file)
print("Exists:", os.path.exists(test_file))

Created: /content/drive/MyDrive/RealBasicVSR_Project/frames/test_write.txt
Exists: True


In [ ]:
import os

test_file = f"{FRAMES_DIR}/test_write.txt"

if os.path.exists(test_file):
    os.remove(test_file)

print("Test file removed:", not os.path.exists(test_file))

Test file removed: True


In [ ]:
!rm -rf /content/test_frames_10
!mkdir -p /content/test_frames_10
!find /content/test_frames -maxdepth 1 -name '*.png' | sort | head -10 | xargs -I{} cp "{}" /content/test_frames_10/

In [ ]:
!sed 's#input_dir = "/content/test_frames"#input_dir = "/content/test_frames_10"#' /content/RealBasicVSR/inference_drive_resume.py > /content/RealBasicVSR/test_drive_10.py

!/content/rbvsr_legacy/bin/python /content/RealBasicVSR/test_drive_10.py

/content/rbvsr_legacy/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
2026-09-29 23:23:34,706 - mmedit - INFO - load checkpoint from torchvision path: torchvision://vgg19
load checkpoint from local path: /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
Total frames: 10
AI Upscaling: 100% 10/10 [00:00<00:00, 2873.20it/s]
DONE: AI frames saved to Drive


In [ ]:
!/content/rbvsr_legacy/bin/python /content/RealBasicVSR/test_drive_10.py

/content/rbvsr_legacy/bin/python: can't open file '/content/RealBasicVSR/test_drive_10.py': [Errno 2] No such file or directory


In [ ]:
!/content/rbvsr_legacy/bin/python /content/RealBasicVSR/inference_drive_resume.py

/content/rbvsr_legacy/bin/python: can't open file '/content/RealBasicVSR/inference_drive_resume.py': [Errno 2] No such file or directory


In [ ]:
!find "$FRAMES_DIR" -maxdepth 1 -type f -name '*.png' | wc -l

find: ‘/content/drive/MyDrive/RealBasicVSR_Project/frames’: No such file or directory
0


In [ ]:
VIDEO_TEMP="/content/rbvsr_video_no_audio.mp4"

!ffmpeg -y -framerate 25 -i "$FRAMES_DIR/%05d.png" -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p "$VIDEO_TEMP"

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
# STEP 5 — Repo, config, checkpoint
!rm -rf /content/RealBasicVSR
!git clone -q https://github.com/ckkelvinchan/RealBasicVSR.git /content/RealBasicVSR
!mkdir -p /content/RealBasicVSR/configs /content/RealBasicVSR/checkpoints
!wget -q https://raw.githubusercontent.com/ckkelvinchan/RealBasicVSR/master/configs/realbasicvsr_x4.py -O /content/RealBasicVSR/configs/realbasicvsr_x4.py
!pip install -q gdown
!gdown 1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID -O /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
!ls -lh /content/RealBasicVSR/checkpoints/RealBasicVSR.pth

Downloading...
From (original): https://drive.google.com/uc?id=1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID
From (redirected): https://drive.google.com/uc?id=1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID&confirm=t&uuid=cc7f6624-2ea9-470a-bafd-ea2ba946bd2a
To: /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
100% 210M/210M [00:02<00:00, 76.0MB/s]
-rw-r--r-- 1 root root 201M Nov 30  2021 /content/RealBasicVSR/checkpoints/RealBasicVSR.pth


In [ ]:
# STEP 6 — Verify
!/content/rbvsr_legacy/bin/python - <<'PY'
import torch, torchvision, numpy, cv2, mmcv, mmedit
print("Python:", __import__("sys").version.split()[0])
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Torchvision:", torchvision.__version__)
print("NumPy:", numpy.__version__)
print("MMCV:", mmcv.__version__)
print("MMEditing:", mmedit.__version__)
print("OpenCV:", cv2.__version__)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
PY

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')


ModuleNotFoundError: No module named 'mmcv'

In [ ]:
# STEP 7 — Create resumable inference script
script = r'''
import os, cv2, torch
from tqdm import tqdm
from mmcv import Config
from mmedit.models import build_model

INPUT_VIDEO=os.environ["RBVSR_INPUT"]
OUT_DIR=os.environ["RBVSR_FRAMES"]
CFG="/content/RealBasicVSR/configs/realbasicvsr_x4.py"
CKPT="/content/RealBasicVSR/checkpoints/RealBasicVSR.pth"
os.makedirs(OUT_DIR, exist_ok=True)

cfg=Config.fromfile(CFG)
model=build_model(cfg.model, train_cfg=None, test_cfg=cfg.get("test_cfg"))
model=model.cuda().eval()
checkpoint=torch.load(CKPT,map_location="cpu")
model.load_state_dict(checkpoint.get("state_dict",checkpoint),strict=False)

cap=cv2.VideoCapture(INPUT_VIDEO)
total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps=cap.get(cv2.CAP_PROP_FPS)
print(f"Total frames: {total}, FPS: {fps}")

for i in tqdm(range(total),desc="AI frames → Google Drive"):
    out_path=os.path.join(OUT_DIR,f"{i+1:05d}.png")
    if os.path.exists(out_path) and os.path.getsize(out_path)>1000:
        continue
    cap.set(cv2.CAP_PROP_POS_FRAMES,i)
    ok,frame=cap.read()
    if not ok: raise RuntimeError(f"Could not read frame {i+1}")
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    img=torch.from_numpy(rgb).float().permute(2,0,1).unsqueeze(0).unsqueeze(0).cuda()/255.0
    with torch.no_grad(): out=model(img,test_mode=True)
    result=out["output"] if isinstance(out,dict) else out
    if result.dim()==5: result=result[0,0]
    elif result.dim()==4: result=result[0]
    result=result.clamp(0,1).permute(1,2,0).cpu().numpy()
    bgr=cv2.cvtColor((result*255).round().astype("uint8"),cv2.COLOR_RGB2BGR)
    if not cv2.imwrite(out_path,bgr): raise RuntimeError(f"Failed to save {out_path}")
cap.release()
print("AI frame generation complete.")
'''
open("/content/RealBasicVSR/inference_drive_resume.py","w").write(script)
print("Inference script ready.")

Inference script ready.


In [ ]:
# STEP 8 — AI processing with Drive checkpoint/resume
import os
os.environ["RBVSR_INPUT"]=INPUT_VIDEO
os.environ["RBVSR_FRAMES"]=FRAMES_DIR
!RBVSR_INPUT="$RBVSR_INPUT" RBVSR_FRAMES="$RBVSR_FRAMES" /content/rbvsr_legacy/bin/python /content/RealBasicVSR/inference_drive_resume.py

Traceback (most recent call last):
  File "/content/RealBasicVSR/inference_drive_resume.py", line 4, in <module>
    from mmcv import Config
ModuleNotFoundError: No module named 'mmcv'


In [ ]:
# STEP 9 — Verify all frames before encoding
import glob, os
frames=sorted(glob.glob(FRAMES_DIR+"/*.png"))
print("Frame count on Drive:",len(frames))
if not frames:
    raise RuntimeError("No PNG frames found.")
print("First:",os.path.basename(frames[0]))
print("Last :",os.path.basename(frames[-1]))

Frame count on Drive: 0


RuntimeError: No PNG frames found.

In [ ]:
# STEP 10 — Copy frames temporarily to local storage for fast FFmpeg encoding
# Drive remains the permanent checkpoint; this local copy can be recreated after a reset.
!rm -rf /content/rbvsr_frames
!mkdir -p /content/rbvsr_frames
!cp "$FRAMES_DIR"/*.png /content/rbvsr_frames/

cp: cannot stat '/content/drive/MyDrive/RealBasicVSR_Project/frames/*.png': No such file or directory


In [ ]:
# STEP 11 — Encode the 4x video
!mkdir -p /content/rbvsr_results
!ffmpeg -y -framerate 25 -i /content/rbvsr_frames/%05d.png -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p /content/rbvsr_results/upscaled_video.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
# STEP 12 — Add original audio
!ffmpeg -y -i /content/rbvsr_results/upscaled_video.mp4 -i "$INPUT_VIDEO" -map 0:v:0 -map 1:a:0? -c:v copy -c:a copy -shortest /content/rbvsr_results/final_4x.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
# STEP 13 — Verify final file
!ffprobe -v error -show_entries format=duration,size -show_entries stream=index,codec_name,width,height,r_frame_rate,codec_type -of default=noprint_wrappers=1 /content/rbvsr_results/final_4x.mp4

/content/rbvsr_results/final_4x.mp4: No such file or directory


In [ ]:
# STEP 14 — Save final MP4 to Google Drive
FINAL_DRIVE=f"{RESULTS_DIR}/final_4x.mp4"
!cp /content/rbvsr_results/final_4x.mp4 "$FINAL_DRIVE"
print("Saved:", FINAL_DRIVE)
!ls -lh "$FINAL_DRIVE"

cp: cannot stat '/content/rbvsr_results/final_4x.mp4': No such file or directory
Saved: /content/drive/MyDrive/RealBasicVSR_Project/results/final_4x.mp4
ls: cannot access '/content/drive/MyDrive/RealBasicVSR_Project/results/final_4x.mp4': No such file or directory


## Recovery rule

যদি Step 8 চলাকালীন runtime disconnect/reset হয়:

- Drive-এর `frames` folder-এ যত PNG তৈরি হয়েছে সেগুলো থাকবে।
- নতুন runtime-এ Notebook আবার চালান।
- একই Drive project folder ব্যবহার করুন।
- Step 8 আবার চালালে existing PNG **skip** হবে এবং missing frame-গুলো তৈরি হবে।
- সব frame পাওয়া গেলে Step 9 থেকে আবার এগিয়ে ভিডিও তৈরি করুন।

**Frame checkpoint এবং final MP4 দুটোই Google Drive-এ থাকবে।**